# YOLO11 semantic 초기 학습

YOLO11s backbone과 semantic decoder를 학습합니다. DINO control(e6)과 specialist·Eu 완료 export가 필요합니다.


## 설정


In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
PROJECT_ROOT=Path('/content/drive/MyDrive/KoMaP2026')
DATA_ROOT=Path('/content/komap_advanced_ram_data')
SOURCE_RUN_NAME='dinov3_hard_ablation_control_01'
EXPECTED_SOURCE_SHA256='1361e35f7331e417704b216bb0253ce221c8d69f58b44e7bce0bd05dcdae17b3'
BASE_RUN_NAMES=['specialist_primary_fast_01','specialist_al3ni_fast_01',
                'specialist_eutectic_fast_01','specialist_al_fast_01']
EU_RUN_NAME='advanced_eu_balanced_ram_01'
BASE_WEIGHTS=[1.0,1.0,1.0,1.0]
EPOCHS=60
MIN_EPOCHS=20
EARLY_STOPPING_PATIENCE=10
STOP_AFTER_EPOCH=None
RESUME=True
PATCH_SIZE=480
BATCH_SIZE=None
PATCHES_PER_IMAGE=32
ZOOM_PROBABILITY=0.5
NUM_WORKERS=2
ENCODER_LR=3e-5
DECODER_LR=3e-4
ABSENT_POLICY='ignore'
ARCHITECTURE='yolo11_edge_semantic'
RUN_NAME='metal_yolo11_edge_semantic_ram_01'


## 환경·코드


In [ ]:
import os, sys, importlib
from pathlib import Path
_candidates = [Path(os.environ['KOMAP_SUBMISSION_ROOT'])] if os.environ.get('KOMAP_SUBMISSION_ROOT') else []
_candidates += [Path.cwd(), Path('/content/KoMaP2026_Submission_DirectInference_20261015'),
                Path('/content/drive/MyDrive/KoMaP2026_Submission_DirectInference_20261015')]
CODE_ROOT = next((p.resolve() for p in _candidates if (p/'komap_submission_environment.py').is_file() and (p/'komap_submission_infer.py').is_file() and 'CURRENT_EXPORT_SHA256' in (p/'komap_submission_infer.py').read_text(encoding='utf-8')), None)
if CODE_ROOT is None:
    raise FileNotFoundError('제출 코드 폴더를 찾지 못했습니다. 00_제출파일_실행안내.ipynb에서 KOMAP_SUBMISSION_ROOT 경로를 설정하세요.')
import subprocess
subprocess.check_call([sys.executable, str(CODE_ROOT/'komap_submission_environment.py'), '--install'])


## 환경·코드


In [ ]:
import os, sys, importlib
from pathlib import Path
_candidates = [Path(os.environ['KOMAP_SUBMISSION_ROOT'])] if os.environ.get('KOMAP_SUBMISSION_ROOT') else []
_candidates += [Path.cwd(), Path('/content/KoMaP2026_Submission_DirectInference_20261015'),
                Path('/content/drive/MyDrive/KoMaP2026_Submission_DirectInference_20261015')]
CODE_ROOT = next((p.resolve() for p in _candidates if (p/'komap_submission_environment.py').is_file() and (p/'komap_submission_infer.py').is_file() and 'CURRENT_EXPORT_SHA256' in (p/'komap_submission_infer.py').read_text(encoding='utf-8')), None)
if CODE_ROOT is None:
    raise FileNotFoundError('제출 코드 폴더를 찾지 못했습니다. 00_제출파일_실행안내.ipynb에서 KOMAP_SUBMISSION_ROOT 경로를 설정하세요.')
CODE_DIR = CODE_ROOT / 'legacy_yolo_initial'
sys.dont_write_bytecode = True

for _module_name in list(sys.modules):
    if _module_name.startswith('komap_'):
        del sys.modules[_module_name]
sys.path[:] = [p for p in sys.path if p not in (str(CODE_ROOT), str(CODE_ROOT/'legacy_yolo_initial'))]
sys.path.insert(0, str(CODE_DIR))
importlib.invalidate_caches()
print('제출 코드:', CODE_DIR)


## 데이터·초기 모델


In [ ]:
import torch
import komap_specialist_data as kd
import komap_phase_fusion as pf
import komap_nieu_zoom_ram as kr
import komap_metal_methods_ram as mm
for module in (kd,pf,kr,mm):
    if Path(module.__file__).resolve().parent!=CODE_DIR.resolve():
        raise RuntimeError('다른 폴더의 코드가 로드되었습니다.')
RAM,DATA_REPORT=kr.prepare_ram(PROJECT_ROOT,DATA_ROOT)
SOURCE_INFO=kr.source_from_completed_control(PROJECT_ROOT,DATA_REPORT,SOURCE_RUN_NAME)
if SOURCE_INFO['sha256']!=EXPECTED_SOURCE_SHA256 or SOURCE_INFO['source_epoch']!=6:
    raise ValueError('기존 최종 조합 기준은 e006 / SHA 1361e35f7331의 control best입니다.')
BASE_EXPORTS=pf.resolve_baseline(PROJECT_ROOT,BASE_RUN_NAMES)
EU_EXPORT=pf.load_export(PROJECT_ROOT/'runs'/EU_RUN_NAME)
BASELINE_CONTEXT=mm.prepare_current_baseline(RAM,DATA_REPORT,SOURCE_INFO,
    BASE_EXPORTS,EU_EXPORT,BASE_WEIGHTS)
PHYSICAL_REPORT=None


## 학습·검증


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError('Colab 런타임 유형을 GPU로 변경하세요.')
ACTUAL_BATCH=BATCH_SIZE if BATCH_SIZE is not None else (8 if torch.cuda.get_device_properties(0).total_memory>=35*2**30 else 4)
CONFIG=mm.Config(architecture=ARCHITECTURE,encoder='resnet34',yolo_preset='yolo11s',
    patch=PATCH_SIZE,batch=ACTUAL_BATCH,patches_per_image=PATCHES_PER_IMAGE,
    zoom_probability=ZOOM_PROBABILITY,zoom_crop=PATCH_SIZE//2,
    epochs=EPOCHS,warmup_epochs=3,freeze_encoder_epochs=1,eval_every=2,
    encoder_lr=ENCODER_LR,decoder_lr=DECODER_LR,eval_stride=PATCH_SIZE//2,
    dropout=0.1,boundary_weight=0.1,boundary_ramp_epochs=5,
    denoise_max_strength=0.25,freeze_encoder_bn=True,
    fixed_new_model_weight=0.25,validation_tta_rotations=4,absent_policy=ABSENT_POLICY)
RUN_DIR=PROJECT_ROOT/'runs'/RUN_NAME
BEST_PATH,BEST_SOLO_PATH=mm.train(RAM,DATA_REPORT,RUN_DIR,CONFIG,BASELINE_CONTEXT,
    num_workers=NUM_WORKERS,resume=RESUME,patience=EARLY_STOPPING_PATIENCE,
    min_epochs=MIN_EPOCHS,stop_after_epoch=STOP_AFTER_EPOCH,physical_report=PHYSICAL_REPORT)
print('고정 결합 best:',BEST_PATH)
print('단독 best:',BEST_SOLO_PATH)


## 확률 내보내기


In [ ]:
EXPORT_DIR=mm.export_candidates(RAM,DATA_REPORT,RUN_DIR,BASELINE_CONTEXT,
    physical_report=PHYSICAL_REPORT)
